<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/02-contract-extraction-pipeline-vs-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study B — Contract extraction: pipeline vs agent

**Goal:** Work the judgment call an interviewer loves (you *can* build it as an agent, but should you?) on a real task (pull fields out of contracts), and settle it with **eval + cost numbers**, not opinion.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **The thesis, up front:** when the steps are known in advance, a **pipeline** beats an **agent**: it's cheaper, testable, and deterministic. The agent's freedom to "decide the next step" is a liability you pay for, not an asset. This notebook builds *both*, on the same task, and measures the difference. (Section [05/03](../05-agents/03-guardrails-and-budgets.ipynb) states the rule; here we prove it.)

## Setup

Self-contained. Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup
client, MODEL = setup()

## The task

A legal-ops team hands you a stack of vendor contracts and wants four fields out of each: **parties**, **effective date**, **annual value**, and **termination notice (days)**. Same four fields, every contract, thousands of them.

Here's the toy corpus and the golden answers (a real one has thousands; the shape is identical).

In [ ]:
CONTRACTS = [
    {"id": "C1", "text": "This Agreement is made between Acme Corp and Globex LLC, effective January 5, 2024. "
                         "The total annual fee is $120,000. Either party may terminate with 30 days written notice."},
    {"id": "C2", "text": "Effective 2023-11-15, Initech and Umbrella Inc agree to the terms herein. "
                         "Annual contract value: $85,500. Termination requires 60 days notice."},
    {"id": "C3", "text": "Between Wayne Enterprises and Stark Industries. Commencement date: March 1 2024. "
                         "Yearly amount of $250,000. This contract may be ended upon 90 days notice."},
]
GOLD = {
    "C1": {"parties": ["Acme Corp", "Globex LLC"], "effective_date": "2024-01-05", "annual_value": 120000, "termination_notice_days": 30},
    "C2": {"parties": ["Initech", "Umbrella Inc"], "effective_date": "2023-11-15", "annual_value": 85500,  "termination_notice_days": 60},
    "C3": {"parties": ["Wayne Enterprises", "Stark Industries"], "effective_date": "2024-03-01", "annual_value": 250000, "termination_notice_days": 90},
}
print(f"{len(CONTRACTS)} contracts, 4 fields each — same shape every time. Hold that thought.")

## A shared scorer + a cost meter

Before either approach, the eval and the meter, so "which is better" is a number, not a vibe. We score exact-match on the four fields, and count tokens (Groq's free tier is $0, but tokens are the honest cost proxy: same idea as [01/04](../01-model-apis/04-context-and-caching.ipynb) and [05/03](../05-agents/03-guardrails-and-budgets.ipynb)).

In [ ]:
import json, re

def score(pred, gold):
    """Fraction of the 4 fields that exactly match."""
    ok = 0
    for k in gold:
        v = pred.get(k)
        if k == "parties":
            ok += sorted(map(str.lower, v or [])) == sorted(map(str.lower, gold[k]))
        else:
            ok += (v == gold[k])
    return ok / len(gold)

class Meter:
    """Accumulate token usage across calls, as a stand-in for $ and latency."""
    def __init__(self): self.calls = 0; self.tokens = 0
    def add(self, resp): self.calls += 1; self.tokens += resp.usage.total_tokens
    def __str__(self): return f"{self.calls} calls, {self.tokens} tokens"

## Approach 1 — the agent

The tempting build: give the model tools and let it *decide* how to extract each field. It feels flexible and "smart." We wire the raw loop from [05/01](../05-agents/01-agent-loop-from-scratch.ipynb) with a couple of tools (find a value, do arithmetic) and let it drive.

In [ ]:
# Tools the agent can call. Deliberately generic — the agent decides how to use them.
def find_line(contract_text, keyword):
    hits = [ln for ln in re.split(r'(?<=[.])\s+', contract_text) if keyword.lower() in ln.lower()]
    return hits[0] if hits else "(no line matches)"

TOOLS = [
    {"type": "function", "function": {"name": "find_line",
        "description": "Return the sentence in the contract containing a keyword (e.g. 'terminate', 'effective').",
        "parameters": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]}}},
]

def extract_agent(contract, meter, max_turns=8):
    text = contract["text"]
    messages = [
        {"role": "system", "content": "Extract contract fields. Use find_line to locate info. When done, reply with "
                                       "ONLY a JSON object: {parties:[..], effective_date:'YYYY-MM-DD', "
                                       "annual_value:<int>, termination_notice_days:<int>}."},
        {"role": "user", "content": f"Contract:\n{text}"},
    ]
    for _ in range(max_turns):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, max_tokens=400, temperature=0)
        meter.add(resp)
        msg = resp.choices[0].message
        if resp.choices[0].finish_reason != "tool_calls":
            m = re.search(r'\{.*\}', msg.content or "", re.DOTALL)
            try: return json.loads(m.group()) if m else {}
            except Exception: return {}
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            out = find_line(text, args.get("keyword", ""))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {}

agent_meter = Meter()
agent_scores = []
for c in CONTRACTS:
    pred = extract_agent(c, agent_meter)
    agent_scores.append(score(pred, GOLD[c["id"]]))
    print(f"{c['id']}: score {agent_scores[-1]:.2f}  pred={pred}")
print(f"\nAGENT — accuracy {sum(agent_scores)/len(agent_scores):.0%}, cost {agent_meter}")

> **⚠️ Production reality —** every run of the agent can take a *different* number of turns and tool calls. That's non-determinism baked in: two identical contracts can cost different amounts and even extract differently. Watch the token count; it's higher than you'd guess, because each turn re-sends the whole history ([01/04](../01-model-apis/04-context-and-caching.ipynb)).

## Approach 2 — the pipeline

The steps are known: for *every* contract we want the same four fields. So write the steps down. One model call, a fixed schema, deterministic parsing: the pattern from [01/01 structured output](../01-model-apis/01-structured-output.ipynb). No loop, no tool-choice, no re-sent history.

In [ ]:
def extract_pipeline(contract, meter):
    text = contract["text"]
    # One call, one job: return the fixed schema. The "steps" are the schema itself.
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=250, temperature=0,
        messages=[{"role": "system", "content":
                   "Extract these fields from the contract and reply with ONLY a JSON object: "
                   "{parties:[two names], effective_date:'YYYY-MM-DD', annual_value:<integer dollars>, "
                   "termination_notice_days:<integer>}."},
                  {"role": "user", "content": text}])
    meter.add(resp)
    m = re.search(r'\{.*\}', resp.choices[0].message.content or "", re.DOTALL)
    try: return json.loads(m.group()) if m else {}
    except Exception: return {}

pipe_meter = Meter()
pipe_scores = []
for c in CONTRACTS:
    pred = extract_pipeline(c, pipe_meter)
    pipe_scores.append(score(pred, GOLD[c["id"]]))
    print(f"{c['id']}: score {pipe_scores[-1]:.2f}  pred={pred}")
print(f"\nPIPELINE — accuracy {sum(pipe_scores)/len(pipe_scores):.0%}, cost {pipe_meter}")

## The verdict — same task, measured

In [ ]:
a_acc, p_acc = sum(agent_scores)/len(agent_scores), sum(pipe_scores)/len(pipe_scores)
print(f"{'':10} {'accuracy':>10} {'calls':>7} {'tokens':>8}")
print(f"{'agent':10} {a_acc:>9.0%} {agent_meter.calls:>7} {agent_meter.tokens:>8}")
print(f"{'pipeline':10} {p_acc:>9.0%} {pipe_meter.calls:>7} {pipe_meter.tokens:>8}")
print(f"\npipeline used {agent_meter.tokens/max(pipe_meter.tokens,1):.1f}x fewer tokens "
      f"for {'better' if p_acc>=a_acc else 'comparable'} accuracy — on a task whose steps never change.")

> **🚩 Common mistake —** a large share of shipped "agents" are pipelines wearing a trench coat: the model "decides" among steps that always run in the same order, and the team pays agent costs (non-determinism, per-step spend, eval difficulty, re-sent history) for pipeline behavior. Extraction is the textbook case: the four fields are fixed, so the "decisions" are fake.

> **🔵 Interview signal —** "we built it as a pipeline because the steps were known, and here's the token/accuracy comparison against the agent version" is a *stronger* answer than "we built an agent." It shows you know **when not to** reach for the loop, the exact judgment section [05/03](../05-agents/03-guardrails-and-budgets.ipynb) is about.

## When the agent *is* right

This case argues for the pipeline **because the steps are known**. Flip that and the agent wins:

- **The path depends on what you find.** "Read this error, decide what to inspect next, repeat" is a debugging or research task where step N+1 genuinely isn't known until step N returns.
- **Open-ended tool use.** The set of actions and their order varies per input in ways you can't enumerate.
- **One branch inside a pipeline.** Often the right shape is a *pipeline* with a single small *bounded agent* promoted only for the one step that truly branches, not an agent wrapping the whole thing.

> **⭐ Key takeaway —** "pipeline vs agent" isn't about sophistication; it's about whether the steps are knowable in advance. Known → pipeline (cheaper, testable, deterministic). Genuinely dynamic → a bounded agent, for *that part only*. Start with the pipeline; promote a step to an agent when you have evidence it needs one. The reverse migration, un-agenting a flaky production system, is far more painful.

## Exercises

1. **Make the agent look good.** Construct a contract where the pipeline's single call *fails* but an agent that can call `find_line` would recover (e.g. fields buried in boilerplate). Does the agent actually win, and at what token cost? This is the honest test of when the loop earns its price.
2. **Promote one step.** Keep the pipeline, but add a *single* bounded agent step only for `annual_value` when the amount is written in words ("two hundred fifty thousand"). Show the hybrid beats both pure versions on a mixed corpus.
3. **The cost at scale.** Using the token counts above, estimate the monthly cost difference at 100k contracts/month (assume a real $/token from [01/04](../01-model-apis/04-context-and-caching.ipynb)). Write the one sentence you'd put in a design doc to justify the pipeline.
4. **Determinism test.** Run the agent extractor on the same contract 5 times and the pipeline 5 times (temperature 0 for both). Which one varies, and why does that matter for testing and on-call?